# 2．验证集 Student–Teacher 对照分析

使用已选定的 **三组 × 三种子全部九个 checkpoint**，对 `val_select` 的 400 条样本推理。直接复用交付包中的冻结 CLIP 特征，不需要图片下载、重新训练或重新选择 checkpoint，CPU 即可运行。

输出：
- `merged_val_select.csv`：3600 行，每行是一个样本 × 一组方法 × 一个种子，保留原始及校准 Teacher 的预测、概率和错误类型。
- `error_type_counts.csv`：按 Teacher 版本、方法、种子及 ALL/E/N/C 分层的完整错误统计，含零计数与分母。
- `representative_cases.csv` 和 `.html`：默认 24 个不重复 sample_id 的代表案例，包含九组 Student 预测及抽样原因。
- 补充指标表、20261011 配对诊断、运行来源哈希和校验记录。

**使用边界：**只分析 val_select，不读取测试报告、测试预测或测试特征；不根据测试错误选择下一轮参数。本验证集已用于 checkpoint 选择，因此这里属于描述性诊断，不能作为独立泛化验证。案例按预设规则分层抽取，不代表错误发生率；错误比例以完整统计表为准。


## 1．环境

需要 Python 3.10+、PyTorch 2.6+、NumPy、pandas、IPython。Colab 通常已包含这些依赖。若下面导入报错，在另一个代码格运行 `%pip install "torch>=2.6" numpy pandas packaging`，然后重启内核并重新运行。


In [ ]:
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone, timedelta
import hashlib, json, zipfile, uuid, shutil, sys, html
import numpy as np
import pandas as pd
import torch
from packaging.version import Version
from IPython.display import display
if Version(torch.__version__.split('+')[0]) < Version('2.6'):
    raise RuntimeError('请先升级到 PyTorch >= 2.6。')
LABELS = ['E', 'N', 'C']  # E=0, N=1, C=2
METHODS = ['CE', 'KD_raw', 'KD_calibrated']
SEEDS = [20261010, 20261011, 20261012]
ERROR_TYPES = ['Teacher对_Student错', '两者都错', 'Teacher错_Student对', '两者都对']
def check(ok, message):
    if not bool(ok):
        raise ValueError(message)
def sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024*1024), b''):
            h.update(block)
    return h.hexdigest()
def read_json(p):
    return json.loads(Path(p).read_text(encoding='utf-8-sig'))
def read_jsonl(p):
    return [json.loads(s) for s in Path(p).read_text(encoding='utf-8').splitlines() if s.strip()]


## 2．输入路径与独立输出目录

Windows 本地运行时，下方默认路径已对应你提供的文件；**不需要提前解压 ZIP**。若已有解压文件夹，可把 `SOURCE` 改为直接包含 students/features/handoff 的目录。

在 Colab 中，请先挂载 Drive，将 ZIP 和整个 Teacher 文件夹上传到 Drive，再将两个路径改成 `/content/drive/MyDrive/...`。Windows 的 C 盘路径不能在 Colab 中使用。

每次运行建立新的目录，不覆盖 v1 或旧分析。ZIP 只提取本分析必需的验证集文件。Teacher CSV 原文件保持只读。


In [ ]:
SOURCE = Path('C:/Users/xhyha/xwechat_files/wxid_raynutuxt57v22_afa2/msg/file/2026-10/student_delivery.zip')
TEACHER_CSV = Path('C:/Users/xhyha/Desktop/Teacher验证集400条_20261012/Teacher_val_select_400.csv')
OUTPUT_ROOT = Path.cwd() / 'val_select_analysis'
CASE_COUNT = 24
check(20 <= CASE_COUNT <= 30, '代表案例数量应为 20–30。')
check(SOURCE.exists() and TEACHER_CSV.is_file(), '输入路径不存在；Colab 请改为 Drive 路径。')
run_id = datetime.now(timezone(timedelta(hours=8))).strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:10]
OUT = OUTPUT_ROOT / run_id
OUT.mkdir(parents=True, exist_ok=False)
required = ['students/selection.json', 'students/training_lock.json',
            'features/val_select.npz', 'features/val_select_meta.json', 'features/clip_revision.json',
            'handoff/manifests/val_select.jsonl', 'handoff/predictions/val_select/jev.jsonl',
            'handoff/results/calibration.json']
if SOURCE.is_file():
    WORK = OUT / '_validation_inputs'
    with zipfile.ZipFile(SOURCE) as z:
        names = z.namelist()
        check(len(names) == len(set(names)), 'ZIP 含重复文件名。')
        check(all(n in names for n in required), 'ZIP 根目录结构不匹配或缺少验证集文件。')
        selection = json.loads(z.read('students/selection.json'))
        checkpoint_names = [c['checkpoint'] for c in selection['configs']]
        check(all(PurePosixPath(n).name == n and chr(92) not in n and ':' not in n for n in checkpoint_names), 'checkpoint 文件名不合法。')
        required += ['students/' + n for n in checkpoint_names]
        for name in required:
            dest = WORK / name
            dest.parent.mkdir(parents=True, exist_ok=True)
            with z.open(name) as src, dest.open('xb') as dst:
                shutil.copyfileobj(src, dst)
else:
    WORK = SOURCE
    selection = read_json(WORK / 'students/selection.json')
    checkpoint_names = [c['checkpoint'] for c in selection['configs']]
    check(all(PurePosixPath(n).name == n and chr(92) not in n and ':' not in n for n in checkpoint_names), 'checkpoint 文件名不合法。')
    required += ['students/' + n for n in checkpoint_names]
check(all((WORK / n).is_file() for n in required), '缺少模型或验证集文件。')
source_hashes = {n: sha(WORK / n) for n in required}
source_hashes['Teacher_val_select_400.csv'] = sha(TEACHER_CSV)
print('本次独立输出目录：', OUT)


## 3．严格核对 sample_id、标签、Teacher 和模型来源

禁止按行号拼接，禁止重复或缺失 sample_id。检查两侧图片、文本、标签和 val_select 划分一致，核对 Teacher 概率与交付包的 Jev logits/校准参数。任一检查失败即停止。


In [ ]:
configs = selection['configs']
expected = {(m, s) for m in METHODS for s in SEEDS}
check(len(configs) == 9 and {(c['method'], c['seed']) for c in configs} == expected, '不是预定的九组模型。')
check(len(set(checkpoint_names)) == 9, 'checkpoint 文件重复。')
for c in configs:
    check(sha(WORK / 'students' / c['checkpoint']) == c['checkpoint_sha256'], 'checkpoint 哈希不匹配。')
training = read_json(WORK / 'students/training_lock.json')
meta = read_json(WORK / 'features/val_select_meta.json')
check(meta['npz_sha256'] == sha(WORK / 'features/val_select.npz') == selection['feature_hashes']['val_select'] == training['feature_hashes']['val_select'], '特征哈希不匹配。')
check(meta['manifest_sha256'] == sha(WORK / 'handoff/manifests/val_select.jsonl'), '清单哈希不匹配。')
check(meta['revision'] == read_json(WORK / 'features/clip_revision.json')['revision'], 'CLIP revision 不一致。')
calibration_path = WORK / 'handoff/results/calibration.json'
check(sha(calibration_path) == selection['calibration_sha256'] == training['calibration_sha256'], '校准来源不匹配。')
manifest = pd.DataFrame(read_jsonl(WORK / 'handoff/manifests/val_select.jsonl'))
teacher = pd.read_csv(TEACHER_CSV, encoding='utf-8-sig', dtype={'sample_id': str, 'image_id': str}, keep_default_na=False)
check(len(manifest) == len(teacher) == 400, '预期为 400 条验证样本。')
check(manifest.sample_id.is_unique and teacher.sample_id.is_unique, 'sample_id 重复。')
check(teacher.sample_id.ne('').all() and set(manifest.sample_id) == set(teacher.sample_id), 'sample_id 缺失或集合不同。')
teacher = teacher.set_index('sample_id').loc[manifest.sample_id].reset_index()
for col in ['image_id', 'image_file', 'hypothesis', 'split', 'source_split']:
    check(manifest[col].astype(str).tolist() == teacher[col].astype(str).tolist(), f'{col} 不一致。')
check(manifest.split.eq('val_select').all(), '只能分析 val_select。')
y = manifest.label.to_numpy(dtype=int)
check(np.isin(y, [0, 1, 2]).all() and np.array_equal(y, teacher.label.to_numpy(dtype=int)), '标签不一致。')
gold = np.array(LABELS)[y]
check(np.array_equal(gold, teacher.gold_label), 'E/N/C 映射不一致。')
with np.load(WORK / 'features/val_select.npz', allow_pickle=False) as data:
    check(data['ids'].tolist() == manifest.sample_id.tolist(), '特征 sample_id 顺序不一致。')
    check(np.array_equal(data['y'], y), '特征标签不一致。')
    check(data['image_ids'].astype(str).tolist() == manifest.image_id.astype(str).tolist(), '特征图片 ID 不一致。')
    x = data['x'].astype(np.float32)
check(x.ndim == 2 and len(x) == 400 and np.isfinite(x).all(), '特征无效。')
def softmax(z):
    e = np.exp(z - z.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)
jev = pd.DataFrame(read_jsonl(WORK / 'handoff/predictions/val_select/jev.jsonl'))
check(jev.sample_id.is_unique and set(jev.sample_id) == set(manifest.sample_id), 'Jev 原始预测 ID 不一致。')
jev = jev.set_index('sample_id').loc[manifest.sample_id]
check(np.array_equal(jev.label.to_numpy(dtype=int), y), 'Jev 标签不一致。')
logits = np.array(jev.logits.tolist(), dtype=float)
params = read_json(calibration_path)['models']['jev']['selected_params']
print('Teacher 校准参数：', params)


下格按已有项目中的校准公式验证 Teacher CSV。原始 Jev 和校准 Jev 分开对照同一批 Student，不混成一个 Teacher；CE 同样保留两种对照。

In [ ]:
# 项目校准公式：softmax((logits + [b, 0, 0]) / T)。
check('b' in params and 'T' in params, '校准参数结构改变，请核对原始 study.py。')
adjusted = logits.copy()
adjusted[:, 0] += float(params['b'])
adjusted /= float(params['T'])
for version, z in [('raw', logits), ('calibrated', adjusted)]:
    columns = [f'jev_{version}_p_{k}' for k in LABELS]
    p = teacher[columns].to_numpy(dtype=float)
    check(np.isfinite(p).all() and ((p >= 0) & (p <= 1)).all(), 'Teacher 概率无效。')
    check(np.allclose(p.sum(axis=1), 1, atol=1e-8), 'Teacher 概率和不为 1。')
    check(np.allclose(p, softmax(z), rtol=1e-6, atol=1e-8), 'Teacher CSV 与交付包来源不一致。')
    pred = np.array(LABELS)[p.argmax(axis=1)]
    check(np.array_equal(pred, teacher[f'jev_{version}_prediction']), 'Teacher 预测与概率不一致。')
    check(np.array_equal((pred == gold).astype(int), teacher[f'jev_{version}_correct'].to_numpy(dtype=int)), 'Teacher 正误标记不一致。')
print('400 个 sample_id、特征、九个模型及 Teacher 来源校验通过。')


## 4．九个 checkpoint 逐一推理

固定使用 CPU 与 eval 模式，关闭 dropout，不更新模型。保留每组概率及 checkpoint 信息；不按验证结果剔除任何种子。

In [ ]:
torch.set_num_threads(max(1, min(4, torch.get_num_threads())))
student_parts = []
for method in METHODS:
    for seed in SEEDS:
        c = next(c for c in configs if c['method'] == method and c['seed'] == seed)
        check(c['dimension'] == x.shape[1], '模型输入维数与特征不一致。')
        model = torch.nn.Sequential(torch.nn.Linear(c['dimension'], 512), torch.nn.ReLU(),
                                    torch.nn.Dropout(.1), torch.nn.Linear(512, 3))
        state = torch.load(WORK / 'students' / c['checkpoint'], map_location='cpu', weights_only=True)
        model.load_state_dict(state, strict=True)
        model.eval()
        with torch.inference_mode():
            z = model(torch.from_numpy(x)).numpy().astype(float)
        p = softmax(z)
        check(np.isfinite(p).all(), 'Student 概率包含无效值。')
        part = pd.DataFrame({'sample_id': manifest.sample_id, 'method': method, 'seed': seed,
                             'checkpoint': c['checkpoint'], 'selected_epoch': c['epoch'],
                             'student_prediction': np.array(LABELS)[p.argmax(axis=1)]})
        for i, label in enumerate(LABELS):
            part[f'student_p_{label}'] = p[:, i]
            part[f'student_logit_{label}'] = z[:, i]
        part['student_confidence'] = p.max(axis=1)
        student_parts.append(part)
student = pd.concat(student_parts, ignore_index=True)
check(len(student) == 3600 and not student.duplicated(['sample_id', 'method', 'seed']).any(), '九组预测不完整或重复。')
merged = student.merge(teacher, on='sample_id', how='left', validate='many_to_one', indicator=True)
check(len(merged) == 3600 and merged['_merge'].eq('both').all(), '合并漏行。')
merged = merged.drop(columns='_merge')
merged['student_correct'] = merged.student_prediction.eq(merged.gold_label)
for version in ['raw', 'calibrated']:
    tc = merged[f'jev_{version}_prediction'].eq(merged.gold_label)
    sc = merged.student_correct
    merged[f'error_type_{version}'] = np.select([tc & ~sc, ~tc & ~sc, ~tc & sc], ERROR_TYPES[:3], default=ERROR_TYPES[3])
    merged[f'both_wrong_same_label_{version}'] = (~tc & ~sc & merged[f'jev_{version}_prediction'].eq(merged.student_prediction))
print('合并完成：', len(merged), '行；400 条 × 9 个模型。')


## 5．错误统计、N 类与 20261011 配对诊断

错误占比的分母是本行所标注的 ALL 或真实标签子集样本数；ALL 与 E/N/C 层不能相加。每个 Teacher 版本、方法、种子、分层内四种错误类型数量之和必须等于该层样本数。N 类的 Student 准确率等于 N 类召回率。

20261011 另外比较同一 sample_id 的 CE → KD_raw，以及 KD_raw → KD_calibrated 的正误转换，防止把 Teacher–Student 差异误当成蒸馏掉分原因。

In [ ]:
counts, metrics_rows = [], []
def macro_f1(gold_labels, predictions):
    values = []
    for label in LABELS:
        tp = ((gold_labels == label) & (predictions == label)).sum()
        fp = ((gold_labels != label) & (predictions == label)).sum()
        fn = ((gold_labels == label) & (predictions != label)).sum()
        denom = 2*tp + fp + fn
        values.append(2*tp/denom if denom else 0.)
    return float(np.mean(values))
for (method, seed), group in merged.groupby(['method', 'seed'], sort=False):
    f1 = macro_f1(group.gold_label.to_numpy(), group.student_prediction.to_numpy())
    selected = next(c for c in configs if c['method'] == method and c['seed'] == seed)
    check(abs(f1 - selected['validation_macro_f1']) < 1e-5, '复现指标与 checkpoint 选择记录不一致。')
    metrics_rows.append(dict(method=method, seed=int(seed), accuracy_pct=100*group.student_correct.mean(),
                             macro_f1_pct=100*f1, N_recall_pct=100*group.loc[group.gold_label.eq('N'), 'student_correct'].mean()))
    for stratum in ['ALL', *LABELS]:
        subset = group if stratum == 'ALL' else group[group.gold_label.eq(stratum)]
        for version in ['raw', 'calibrated']:
            vc = subset[f'error_type_{version}'].value_counts()
            for error in ERROR_TYPES:
                count = int(vc.get(error, 0))
                counts.append(dict(teacher_version=version, method=method, seed=int(seed), gold_stratum=stratum,
                                   error_type=error, count=count, denominator=len(subset),
                                   pct=100*count/len(subset) if len(subset) else float('nan')))
stats = pd.DataFrame(counts)
check(stats.groupby(['teacher_version','method','seed','gold_stratum'])['count'].sum().tolist() ==
      stats.groupby(['teacher_version','method','seed','gold_stratum']).denominator.first().tolist(), '计数无法闭合。')
metrics_table = pd.DataFrame(metrics_rows)
predwide = merged.pivot(index='sample_id', columns=['method', 'seed'], values='student_prediction')
base = teacher.set_index('sample_id').copy()
for method in METHODS:
    for seed in SEEDS:
        base[f'{method}_{seed}_prediction'] = predwide[(method, seed)]
focus = []
for before, after in [('CE', 'KD_raw'), ('KD_raw', 'KD_calibrated')]:
    for stratum in ['ALL', *LABELS]:
        g = base if stratum == 'ALL' else base[base.gold_label.eq(stratum)]
        a = g[f'{before}_20261011_prediction'].eq(g.gold_label)
        b = g[f'{after}_20261011_prediction'].eq(g.gold_label)
        focus.append(dict(seed=20261011, comparison=f'{after} minus {before}', gold_stratum=stratum,
                          denominator=len(g), correct_to_wrong=int((a & ~b).sum()), wrong_to_correct=int((~a & b).sum()),
                          both_correct=int((a & b).sum()), both_wrong=int((~a & ~b).sum()),
                          accuracy_delta_pp=100*(b.mean()-a.mean())))
focus_table = pd.DataFrame(focus)
display(metrics_table.round(3))
display(stats[stats.gold_stratum.eq('N') & stats.seed.eq(20261011)])
display(focus_table.round(3))


## 6．选出 24 条代表案例（不重复 sample_id）

确定性分层规则：先取 N 类、KD_raw/20261011 对原始 Teacher 的三种错误各最多 4 条；再取该种子 CE 对但 KD_raw 错的 N 类样本最多 4 条、校准 KD 修复原始 KD 错误的 N 类最多 4 条；再从其余九组、两个 Teacher 版本的三种错误补足。每个候选池按 sample_id 的固定哈希排序，不按置信度挑最极端案例；不足时如实补充并记录抽样原因。每张图片优先至多一例，不足时才放宽图片去重。

案例同时展示九组 Student 预测和两个 Teacher，便于比较。仅凭预测不能认定视觉、语义或标签错误原因；保留 hypothesis 和 image_file，人工查看原图后填写 `人工备注`。这里不自动编造图像内容。


In [ ]:
focus_raw = merged[(merged.method == 'KD_raw') & (merged.seed == 20261011)].set_index('sample_id')
base['focus_error_raw'] = focus_raw['error_type_raw']
base['focus_error_calibrated'] = focus_raw['error_type_calibrated']
base['focus_student_confidence'] = focus_raw['student_confidence']
chosen, reasons, used_images = [], {}, set()
def choose(ids, quota, reason, distinct_image=True):
    ordered = sorted(set(ids), key=lambda sid: hashlib.sha256(('val_cases_v1|' + sid).encode()).hexdigest())
    added = 0
    for sid in ordered:
        if len(chosen) >= CASE_COUNT or added >= quota:
            break
        if sid in reasons or (distinct_image and str(base.loc[sid, 'image_id']) in used_images):
            continue
        chosen.append(sid); reasons[sid] = reason; added += 1
        used_images.add(str(base.loc[sid, 'image_id']))
is_n = base.gold_label.eq('N')
for error in ERROR_TYPES[:3]:
    choose(base.index[is_n & base.focus_error_raw.eq(error)], 4, 'N类 / KD_raw 20261011 / 原始' + error)
ce_ok = base.CE_20261011_prediction.eq(base.gold_label)
raw_ok = base.KD_raw_20261011_prediction.eq(base.gold_label)
cal_ok = base.KD_calibrated_20261011_prediction.eq(base.gold_label)
choose(base.index[is_n & ce_ok & ~raw_ok], 4, 'N类 / 20261011 CE对但原始KD错')
choose(base.index[is_n & ~raw_ok & cal_ok], 4, 'N类 / 20261011 校准KD修复原始KD错误')
# 每池每轮最多补一例，兼顾其它种子、方法与错误类型。
pools = []
for error in ERROR_TYPES[:3]:
    for seed in SEEDS:
        for method in METHODS:
            for version in ['raw', 'calibrated']:
                g = merged[(merged.seed == seed) & (merged.method == method) & merged[f'error_type_{version}'].eq(error)]
                pools.append((g.sample_id.tolist(), f'补充 / {method} {seed} / {version} / {error}'))
for _ in range(CASE_COUNT):
    size = len(chosen)
    for ids, reason in pools:
        choose(ids, 1, reason)
    if len(chosen) == size or len(chosen) >= CASE_COUNT:
        break
if len(chosen) < CASE_COUNT:
    for ids, reason in pools:
        choose(ids, CASE_COUNT-len(chosen), reason + ' / 放宽图片去重', distinct_image=False)
if len(chosen) < CASE_COUNT:
    choose(base.index, CASE_COUNT-len(chosen), '补充对照样本（错误候选不足）', distinct_image=False)
cases = base.loc[chosen].reset_index()
cases.insert(0, 'case_no', range(1, len(cases)+1))
cases.insert(1, 'selection_reason', [reasons[s] for s in chosen])
cases['人工备注'] = ''
check(len(cases) == CASE_COUNT and cases.sample_id.is_unique, '代表案例数量或去重失败。')
display(cases[['case_no', 'selection_reason', 'sample_id', 'gold_label', 'hypothesis',
               'jev_raw_prediction', 'jev_calibrated_prediction', 'CE_20261011_prediction',
               'KD_raw_20261011_prediction', 'KD_calibrated_20261011_prediction']])


## 7．保存交付文件与来源记录

CSV 使用 UTF-8 BOM，方便 Excel 打开。HTML 是便于阅读的案例卡片，包含图片文件名但不嵌入未提供的图片。本次完整运行后产生 `COMPLETE.json`；输出 ZIP 仅包含分析结果，不重复打包模型和特征。


In [ ]:
tables = {'merged_val_select.csv': merged, 'error_type_counts.csv': stats,
          'representative_cases.csv': cases, 'student_val_metrics.csv': metrics_table,
          'seed20261011_paired_diagnosis.csv': focus_table}
for name, frame in tables.items():
    with (OUT / name).open('x', encoding='utf-8-sig', newline='') as f:
        frame.to_csv(f, index=False)
cards = []
for _, row in cases.iterrows():
    esc = lambda value: html.escape(str(value))
    predictions = ''.join('<tr><td>' + esc(m) + '</td>' + ''.join('<td>' + esc(row[f'{m}_{s}_prediction']) + '</td>' for s in SEEDS) + '</tr>' for m in METHODS)
    cards.append(f"<article><h2>案例 {row.case_no} · Gold {esc(row.gold_label)}</h2><p>{esc(row.selection_reason)}</p><p>ID: {esc(row.sample_id)}<br>图片: {esc(row.image_file)}</p><blockquote>{esc(row.hypothesis)}</blockquote><p>Teacher 原始: {esc(row.jev_raw_prediction)}；校准: {esc(row.jev_calibrated_prediction)}</p><table><tr><th>Student</th><th>20261010</th><th>20261011</th><th>20261012</th></tr>{predictions}</table><p>人工备注：待核对原图；不从预测推断具体错误原因。</p></article>")
page = '<!doctype html><html lang="zh"><meta charset="utf-8"><title>验证集代表案例</title><style>body{font:16px system-ui;max-width:960px;margin:40px auto;padding:20px;background:#f5f7fa;color:#17263b}article{background:white;padding:24px;margin:24px 0;border-radius:12px}td,th{padding:8px 18px;border:1px solid #ccd4df}table{border-collapse:collapse}blockquote{margin:16px 0;padding:16px;background:#eef3f9}</style><h1>val_select 代表案例</h1><p>24条预设规则抽样（实际数量见下方），仅用于验证集描述性诊断。N=neutral；E=entailment；C=contradiction。</p>' + ''.join(cards) + '</html>'
with (OUT / 'representative_cases.html').open('x', encoding='utf-8') as f:
    f.write(page.replace('24条预设规则抽样', f'{len(cases)}条预设规则抽样'))
for name in required:
    check(source_hashes[name] == sha(WORK / name), '输入在运行期间发生变化。')
check(source_hashes['Teacher_val_select_400.csv'] == sha(TEACHER_CSV), 'Teacher CSV 在运行期间发生变化。')
provenance = dict(split='val_select', samples=400, models=9, merged_rows=len(merged), cases=len(cases),
                  source=str(SOURCE), teacher_csv=str(TEACHER_CSV), input_sha256=source_hashes,
                  versions={'python': sys.version, 'torch': torch.__version__, 'numpy': np.__version__, 'pandas': pd.__version__},
                  inference_device='cpu', teacher_versions=['raw', 'calibrated'], label_order=LABELS,
                  case_policy='N / KD_raw 20261011 priority; fixed sample_id hash; image diversity; round-robin other errors',
                  test_data_used=False, checkpoint_reselection=False,
                  interpretation='val_select already used for checkpoint selection; descriptive diagnosis only')
with (OUT / 'analysis_provenance.json').open('x', encoding='utf-8') as f:
    json.dump(provenance, f, ensure_ascii=False, indent=2)
outputs = [OUT / name for name in tables] + [OUT / 'representative_cases.html', OUT / 'analysis_provenance.json']
with (OUT / 'COMPLETE.json').open('x', encoding='utf-8') as f:
    json.dump({'status': 'complete', 'output_sha256': {p.name: sha(p) for p in outputs}}, f, indent=2)
bundle = OUT / 'Student_Teacher_val_select_analysis.zip'
with zipfile.ZipFile(bundle, 'x', zipfile.ZIP_DEFLATED) as z:
    for p in outputs + [OUT / 'COMPLETE.json']:
        z.write(p, p.name)
print('交付文件已保存：', OUT)
print('可下载分析包：', bundle)
